# Analyze Finger-Tapping Data

NOTE: As there is a delay between the audio playback onset and the time recording onset when the test is distributed online, we only compute the statistics on finger tapping intervals for now. We will fetch the audio playback onset heard by the audience in the future, and adjust the statitical analysis accordingly. 

Stimulis were presented as a group, containing 3 excerpts in Anchor, Music Transformer (MT) and our proposed model (MASS) with same prompts. Since both MASS and MT generate phrases as a continuation given the same prompts, samples in a group should have the same beat onsets. 

## Preprocess
1. **Reject Invalid Group** We reject invalid groups in which the subject tap anchor in an unexpected way,
    * the variance of the tapping interval is larger than 1/2 ground truth beat interval
    * the mean tapping interval doesn't approximate the true beat interval or half/multiples of the beat interval
2. **Removing Anormaly** Remove jiterry tapping by rejecting finger tapping interval smaller than eps (0.01 second).
3. **Beat Asynchronization** Interval asynchornization is the difference between ground truth inter-beat interval and the finger-tapping interval. 

## Analysis
We use F-test to measure whether MT and MASS results in different variance in beat asynchronization, and report FDR-corrected intra-person p-value.

## Ground Truth Beat

In [1]:
import numpy as np
import pandas as pd

eps = 0.01

# Reference/Anchor/Mass Excerpt Info
song_id_list = ['03', '04', '05', '06', '07', '08', '10', '11', '12', '20']
tp_ts = {'03': {"tp": 120, "ts": 4, "n_bar": 8},
         '04': {"tp": 120, "ts": 2, "n_bar": 9},
         '05': {"tp": 120, "ts": 4, "n_bar": 8},
         '06': {"tp": 120, "ts": 4, "n_bar": 8}, 
         '07': {"tp": 132, "ts": 3, "n_bar": 8},
         '08': {"tp": 120, "ts": 4, "n_bar": 8}, 
         '10': {"tp": 108, "ts": 2, "n_bar": 9},
         '11': {"tp": 144, "ts": 4, "n_bar": 8},
         '12': {"tp": 72, "ts": 3, "n_bar": 9},
         '20': {"tp": 96, "ts": 2, "n_bar": 8}}
         # '20': {"tp": 144, "ts": 3, "n_bar": 8}}

true_beat = {"Ref":{}, "MT":{}}
for song_id in song_id_list:
    
    # Ground truth beats in Reference Excerpt
    t_beat = 60/tp_ts[song_id]['tp']
    t_bar = t_beat * tp_ts[song_id]['ts']
    t_end = t_bar * tp_ts[song_id]['n_bar']
    
    # Remove first two bars
    true_beat["Ref"][song_id] = np.round(np.arange(t_bar * 2, t_end + t_beat, t_beat), 6)

    # Ground truth beats in MT Excerpt, not used yet
    mt_beat_file = f"./data/mt_beat/{song_id}.txt"
    mt_beat = pd.read_csv(f"./audio/mt/{song_id}.txt", header=None)[0].to_numpy()
    mt_beat = mt_beat[mt_beat >= t_bar * 2 - eps]
    true_beat["MT"][song_id] = np.round(mt_beat, 6)

## Finger-Tapping Data

In [2]:
import pandas as pd
from copy import deepcopy

# Parse Tapping Data
df = pd.read_csv("./data/Finger Tapping data.csv")

raw_tap_data = {}

for _, row in df.iterrows():
    id = row['id'][:4]
    type, song_id = (row['phrase'].split(".")[0]).split("/")
    
    t_tap = np.array([np.round(float(i)/1000, 6) for i in row['values'][1:-1].split(", ")])
    if id not in raw_tap_data:
        raw_tap_data[id] = {}
        
    if song_id not in raw_tap_data[id]:
        raw_tap_data[id][song_id] = {}

    delay = t_tap[-1] - true_beat['Ref'][song_id][-1]
    raw_tap_data[id][song_id][type] = t_tap[t_tap >= true_beat['Ref'][song_id][0] + delay][:-1]

In [3]:
# Preprocess Tapping Data

eps = 0.01
valid_ratio = [0.5, 1.0, 2.0]
modes = ['Anchor', 'MASS', 'MT']

tap_interval = {}
tap_async = {}

user_id_list = list(raw_tap_data.keys())
for user_id in user_id_list:
    
    tap_interval[user_id] = {}
    tap_async[user_id] = {}
    
    for song_id in raw_tap_data[user_id]:
        
        # Skip incomplete session
        if len(raw_tap_data[user_id][song_id]) < 3:
            continue

        # Reject group with invalid Anchor

        # Reject anchor if std is too large
        anchor_interval = np.diff(raw_tap_data[user_id][song_id]['Anchor'])
        anchor_std = np.std(anchor_interval)
        t_true_beat = 60/tp_ts[song_id]['tp']
        if anchor_std >= t_true_beat/2:
            continue

        # Reject anchor if tapping an unexpect pattern
        t_anchor_avg = np.mean(anchor_interval)
        ratios = [np.round(t_anchor_avg/t_true_beat, 1),  np.round(t_true_beat/t_anchor_avg, 1)]
        
        is_valid = any([(ratio in valid_ratio) for ratio in ratios])
        if not is_valid:
            continue

        # ratio = valid_ratio[np.argmin(np.abs(valid_ratio - t_anchor_avg/t_true_beat))]

        # Tapping Interval
        t_mt = np.diff(raw_tap_data[user_id][song_id]['MT'])
        t_mass = np.diff(raw_tap_data[user_id][song_id]['MASS'])

        # Remove jittery taps
        t_mt = t_mt[t_mt > eps]
        t_mass = t_mass[t_mass > eps]

        # Handle Skipped beats
        mt_async = np.min([np.abs(t_mt - t_true_beat), 
                           np.abs(t_mt - 2 * t_true_beat),
                           np.abs(t_mt - 4 * t_true_beat)], axis=0)
        mass_async = np.min([np.abs(t_mass - t_true_beat), 
                             np.abs(t_mass - 2 * t_true_beat),
                             np.abs(t_mass - 4 * t_true_beat),], axis=0)

        tap_interval[user_id][song_id] = {"MT": t_mt, "MASS": t_mass}
        tap_async[user_id][song_id] = {"MT": mt_async, "MASS": mass_async}

In [4]:
# Concatenate groups for each subject
t_async = {}
for user_id, entry in tap_async.items():
    t_async[user_id] = {"MASS": [], "MT": []}
    for song_id in entry:
        for mode in entry[song_id]:
            t_async[user_id][mode] += deepcopy(list(entry[song_id][mode]))

## Analysis

In [5]:
import scipy.stats
from statsmodels.stats import multitest

def get_f_stats(x, y):
    var_x, var_y = np.var(x), np.var(y)
    F = var_x/var_y
    df1, df2 = len(x) - 1, len(y) - 1
    p_value = 1 - scipy.stats.f.cdf(F, df1, df2)
    return var_x, var_y, p_value

# Get F-Stats
p_values = []
f_values = []
for user_id in tap_async:
    x = t_async[user_id]['MT']
    y = t_async[user_id]['MASS']

    var_x, var_y, p_value = get_f_stats(x, y)

    # print(user_id, var_x, var_y, p_value)
    f_values.append(var_x/var_y)
    p_values.append(p_value)

is_significant, corrected_p_values = multitest.fdrcorrection(p_values)

In [6]:
print("Subject\t\tF-value\t\tp-value")
for i, v in enumerate(corrected_p_values):
    p_value = corrected_p_values[i]
    if np.round(p_value, 4) == 0:
        p_value = corrected_p_values[i]
    else:
        p_value = np.round(p_value, 4)
    print(f"Subject {i}\t{f_values[i]:.6f}\t{p_value}")

Subject		F-value		p-value
Subject 0	2.350123	3.848314103066519e-05
Subject 1	9.276372	3.3306690738754696e-16
Subject 2	11.241379	3.3306690738754696e-16
Subject 3	0.982473	0.532
Subject 4	4.415471	0.0008
Subject 5	3.425022	6.1565641473748656e-09


In [7]:
p_values

[2.5655427353776794e-05,
 1.1102230246251565e-16,
 1.1102230246251565e-16,
 0.5320455829192471,
 0.0006712760586287114,
 3.0782820736874328e-09]